# Exercise 5 - Feature Engineering

**Anggota Kelompok:**
1. M. Furqon Lutfi Septriandri (25031554192)
2. Akram Ali Fi Natarik (25031554212)

**Data yang dipakai (hasil pre-processing Exercise 4):**
1. `detik_preprocessed.csv` - artikel detik.com "50 Kata-kata Bijak Hari Ini" (Bahasa Indonesia, 58 dokumen)
2. `jsonplaceholder_preprocessed.csv` - REST API JSONPlaceholder `/posts` (teks dummy/lorem ipsum, 100 dokumen)

**Teknik feature engineering yang diterapkan** (mengikuti materi Chapter 5 & 6):
1. One-Hot Encoding
2. TF-IDF
3. Word2Vec
4. FastText

## 0. Install & Import Library

In [1]:
import warnings
warnings.filterwarnings(action='ignore')

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

import nltk
from nltk import word_tokenize
from tqdm import tqdm

import gensim.models

from umap import UMAP
import plotly.express as px

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sept\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sept\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 1. Load Data Hasil Pre-processing (Exercise 4)

In [2]:
df_detik = pd.read_csv("detik_preprocessed.csv")
df_detik['prepro'] = df_detik['prepro'].fillna('')
df_detik.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   No      58 non-null     int64
 1   text    58 non-null     str  
 2   prepro  58 non-null     str  
dtypes: int64(1), str(2)
memory usage: 1.5 KB


In [3]:
df_detik.head()

,No,text,prepro
0,1,Kata-kata Bijak Tentang Pertumbuhan Diri,bijak tumbuh
1,2,Kata-kata Bijak Tentang Kebahagiaan & Syukur,bijak bahagia syukur
2,3,Kata-kata Bijak Tentang Perjuangan & Kegagalan,bijak juang gagal
3,4,Kata-kata Bijak Tentang Hubungan Sesama Manusia,bijak hubung manusia
4,5,Kata-kata Bijak Prinsip Hidup Sehari-hari,bijak prinsip hidup sehari


In [4]:
df_api = pd.read_csv("jsonplaceholder_preprocessed.csv")
df_api['prepro'] = df_api['prepro'].fillna('')
df_api.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   userId  100 non-null    int64
 1   id      100 non-null    int64
 2   text    100 non-null    str  
 3   prepro  100 non-null    str  
dtypes: int64(2), str(2)
memory usage: 3.3 KB


In [5]:
df_api.head()

,userId,id,text,prepro
0,1,1,sunt aut facere repellat provident occaecati e...,sunt aut facere repellat provident occaecati e...
1,1,2,qui est esse est rerum tempore vitae\nsequi si...,qui est esse est rerum tempore vitae sequi sin...
2,1,3,ea molestias quasi exercitationem repellat qui...,ea molestias quasi exercitationem repellat qui...
3,1,4,eum et est occaecati ullam et saepe reiciendis...,eum et est occaecati ullam et saepe reiciendis...
4,1,5,nesciunt quas odio repudiandae veniam quaerat ...,nesciunt quas odio repudiandae veniam quaerat ...


## 2. One-Hot Encoding

### 2.1 Dataset detik.com

In [6]:
vectorizer_ohe_detik = CountVectorizer(binary=True)
X_ohe_detik = vectorizer_ohe_detik.fit_transform(df_detik['prepro'])
df_ohe_detik = pd.DataFrame(X_ohe_detik.toarray(), columns=vectorizer_ohe_detik.get_feature_names_out())
df_ohe_detik

,50,ajaib,alah,alam,alih,amanah,ampuh,angkat,anugerah,ayo,...,tulus,tumbuh,tunggu,uang,ubah,ujian,ujung,usia,utama,zona
0,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,1,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
sparsity_detik = 1 - X_ohe_detik.nnz / (X_ohe_detik.shape[0] * X_ohe_detik.shape[1])
print("Shape       :", X_ohe_detik.shape)
print("Sparsity    :", round(sparsity_detik * 100, 1), "%")

Shape       : (58, 166)
Sparsity    : 97.6 %


### 2.2 Dataset JSONPlaceholder

In [8]:
vectorizer_ohe_api = CountVectorizer(binary=True)
X_ohe_api = vectorizer_ohe_api.fit_transform(df_api['prepro'])
df_ohe_api = pd.DataFrame(X_ohe_api.toarray(), columns=vectorizer_ohe_api.get_feature_names_out())
df_ohe_api.head()

,ab,accusamus,accusantium,ad,adipisci,alias,aliquam,aliquid,amet,animi,...,veniam,veritatis,vero,vitae,voluptas,voluptate,voluptatem,voluptates,voluptatibus,voluptatum
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
2,0,0,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,0,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
4,0,0,0,0,0,1,0,0,0,0,...,1,0,0,0,0,0,1,0,1,0


In [9]:
sparsity_api = 1 - X_ohe_api.nnz / (X_ohe_api.shape[0] * X_ohe_api.shape[1])
print("Shape       :", X_ohe_api.shape)
print("Sparsity    :", round(sparsity_api * 100, 1), "%")

Shape       : (100, 181)
Sparsity    : 85.4 %


### Analisis One-Hot Encoding

Hasil (saat dijalankan dengan file CSV di atas):
- **detik**: matriks berukuran **58 dokumen x 166 kata unik**, sparsity **97.6%** (hampir semua sel bernilai 0).
- **jsonplaceholder**: matriks berukuran **100 dokumen x 181 kata unik**, sparsity **85.4%**.

## 3. TF-IDF

### 3.1 Dataset detik.com

In [10]:
vectorizer_tfidf_detik = TfidfVectorizer()
X_tfidf_detik = vectorizer_tfidf_detik.fit_transform(df_detik['prepro'])
df_tfidf_detik = pd.DataFrame(X_tfidf_detik.toarray(), columns=vectorizer_tfidf_detik.get_feature_names_out())
df_tfidf_detik

,50,ajaib,alah,alam,alih,amanah,ampuh,angkat,anugerah,ayo,...,tulus,tumbuh,tunggu,uang,ubah,ujian,ujung,usia,utama,zona
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.810037,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,0.232301,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,0.344386,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.344386,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [11]:
skor_detik = X_tfidf_detik.sum(axis=0).A1
top10_detik = pd.Series(skor_detik, index=vectorizer_tfidf_detik.get_feature_names_out()).sort_values(ascending=False).head(10)
top10_detik

laku       3.131563
bijak      3.052106
orang      3.047509
hidup      3.032866
bahagia    3.032460
syukur     2.524793
lihat      2.098684
gagal      1.643549
dunia      1.426703
cerdas     1.405148
dtype: float64

### 3.2 Dataset JSONPlaceholder

In [12]:
vectorizer_tfidf_api = TfidfVectorizer()
X_tfidf_api = vectorizer_tfidf_api.fit_transform(df_api['prepro'])
df_tfidf_api = pd.DataFrame(X_tfidf_api.toarray(), columns=vectorizer_tfidf_api.get_feature_names_out())
df_tfidf_api.head()

,ab,accusamus,accusantium,ad,adipisci,alias,aliquam,aliquid,amet,animi,...,veniam,veritatis,vero,vitae,voluptas,voluptate,voluptatem,voluptates,voluptatibus,voluptatum
0,0.0,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.0,...,0.00000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0
1,0.0,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.0,...,0.00000,0.0,0.0,0.177802,0.0,0.158018,0.000000,0.0,0.000000,0.0
2,0.0,0.0,0.195513,0.190225,0.00000,0.000000,0.0,0.0,0.000000,0.0,...,0.00000,0.0,0.0,0.000000,0.0,0.000000,0.246923,0.0,0.000000,0.0
3,0.0,0.0,0.000000,0.000000,0.18337,0.000000,0.0,0.0,0.161393,0.0,...,0.00000,0.0,0.0,0.000000,0.0,0.000000,0.250354,0.0,0.000000,0.0
4,0.0,0.0,0.000000,0.000000,0.00000,0.232895,0.0,0.0,0.000000,0.0,...,0.19985,0.0,0.0,0.000000,0.0,0.000000,0.142836,0.0,0.209239,0.0


In [13]:
skor_api = X_tfidf_api.sum(axis=0).A1
top10_api = pd.Series(skor_api, index=vectorizer_tfidf_api.get_feature_names_out()).sort_values(ascending=False).head(10)
top10_api

et            11.484073
aut            7.754887
qui            7.298075
ut             6.693589
est            6.471584
quia           5.975972
voluptatem     5.567400
omnis          5.367130
voluptas       5.225936
sed            5.024737
dtype: float64

### Analisis TF-IDF

10 kata dengan total skor TF-IDF tertinggi:
- **detik**: `laku, bijak, orang, hidup, bahagia, syukur, lihat, gagal, dunia, cerdas` - semuanya kata kunci yang relevan dengan tema situs (kata-kata bijak tentang hidup, kebahagiaan, perjuangan). Ini masuk akal karena TF-IDF memberi bobot lebih tinggi ke kata yang sering muncul di satu dokumen tapi tidak di semua dokumen, jadi kata generik yang sudah dibuang di tahap stopword removal tidak lagi mendominasi.
- **jsonplaceholder**: `et, aut, qui, ut, est, quia, voluptatem, omnis, voluptas, sed` - ini adalah kata-kata umum dalam teks *lorem ipsum* (placeholder Latin yang dipakai developer untuk mengisi konten dummy, bukan bahasa Latin asli yang bermakna).

## 4. Word2Vec

### 4.1 Tokenisasi Kalimat

In [14]:
sentences_detik = [word_tokenize(teks.lower()) for teks in tqdm(df_detik['prepro'])]
sentences_detik[:3]

100%|██████████| 58/58 [00:00<00:00, 960.96it/s]


[['bijak', 'tumbuh'],
 ['bijak', 'bahagia', 'syukur'],
 ['bijak', 'juang', 'gagal']]

In [15]:
sentences_api = [word_tokenize(teks.lower()) for teks in tqdm(df_api['prepro'])]
sentences_api[:2]

100%|██████████| 100/100 [00:00<00:00, 9023.50it/s]


[['sunt',
  'aut',
  'facere',
  'repellat',
  'provident',
  'occaecati',
  'excepturi',
  'optio',
  'reprehenderit',
  'quia',
  'et',
  'suscipit',
  'suscipit',
  'recusandae',
  'consequuntur',
  'expedita',
  'et',
  'cum',
  'reprehenderit',
  'molestiae',
  'ut',
  'ut',
  'quas',
  'totam',
  'nostrum',
  'rerum',
  'est',
  'autem',
  'sunt',
  'rem',
  'eveniet',
  'architecto'],
 ['qui',
  'est',
  'esse',
  'est',
  'rerum',
  'tempore',
  'vitae',
  'sequi',
  'sint',
  'nihil',
  'reprehenderit',
  'dolor',
  'beatae',
  'ea',
  'dolores',
  'neque',
  'fugiat',
  'blanditiis',
  'voluptate',
  'porro',
  'vel',
  'nihil',
  'molestiae',
  'ut',
  'reiciendis',
  'qui',
  'aperiam',
  'non',
  'debitis',
  'possimus',
  'qui',
  'neque',
  'nisi',
  'nulla']]

### 4.2 Build Model Word2Vec - dataset detik

In [16]:
# 1. Train Model
model_w2v_detik = gensim.models.Word2Vec(sentences_detik, min_count=1, vector_size=100, window=5, sg=0)

In [17]:
# 2. Save Model
model_w2v_detik.save('detik.w2v')

In [18]:
# 3. Load Model
model_w2v_detik = gensim.models.Word2Vec.load('detik.w2v')

In [19]:
# 4. Model Information
w2v_detik = model_w2v_detik.wv
w2v_detik.index_to_key[:10]

['hidup',
 'orang',
 'bijak',
 'laku',
 'bahagia',
 'lihat',
 'syukur',
 'tuju',
 'cerdas',
 'senyum']

In [20]:
w2v_detik.vectors.shape, w2v_detik.vector_size # ukuran vektor

((166, 100), 100)

In [21]:
w2v_detik.similar_by_key('bijak')

[('hormat', 0.36978742480278015),
 ('temu', 0.21180470287799835),
 ('jaga', 0.20496925711631775),
 ('angkat', 0.2018902748823166),
 ('cerdas', 0.19921047985553741),
 ('sastra', 0.19690676033496857),
 ('alih', 0.18891648948192596),
 ('sukses', 0.18204930424690247),
 ('dunia', 0.17258131504058838),
 ('manusia', 0.17117734253406525)]

### 4.3 Build Model Word2Vec - dataset jsonplaceholder

In [22]:
model_w2v_api = gensim.models.Word2Vec(sentences_api, min_count=1, vector_size=100, window=5, sg=0)
model_w2v_api.save('jsonplaceholder.w2v')
model_w2v_api = gensim.models.Word2Vec.load('jsonplaceholder.w2v')

w2v_api = model_w2v_api.wv
w2v_api.index_to_key[:10]

['et',
 'aut',
 'qui',
 'ut',
 'est',
 'quia',
 'voluptatem',
 'voluptas',
 'omnis',
 'molestiae']

In [23]:
w2v_api.vectors.shape, w2v_api.vector_size

((182, 100), 100)

In [24]:
w2v_api.similar_by_key('et')

[('qui', 0.9427904486656189),
 ('amet', 0.9392408728599548),
 ('molestiae', 0.9384351372718811),
 ('sint', 0.9381881952285767),
 ('voluptas', 0.9356023669242859),
 ('non', 0.9346427917480469),
 ('aut', 0.9265221953392029),
 ('omnis', 0.9257065653800964),
 ('consequatur', 0.9250317811965942),
 ('ut', 0.9235471487045288)]

### 4.4 Visualisasi Word2Vec

In [25]:
X = UMAP().fit_transform(w2v_detik.vectors)

df2_detik = pd.DataFrame(X, columns=['umap1', 'umap2'])
df2_detik['word'] = w2v_detik.index_to_key

fig = px.scatter(df2_detik, x='umap1', y='umap2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(height=800, title_text='Word2Vec Visualization - detik')
fig.show()

In [26]:
X = UMAP().fit_transform(w2v_api.vectors)

df2_api = pd.DataFrame(X, columns=['umap1', 'umap2'])
df2_api['word'] = w2v_api.index_to_key

fig = px.scatter(df2_api, x='umap1', y='umap2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(height=800, title_text='Word2Vec Visualization - jsonplaceholder')
fig.show()

### Analisis Word2Vec

- **Ukuran korpus terlalu kecil untuk melatih Word2Vec secara ideal.** Word2Vec biasanya butuh korpus jutaan kata agar vektor yang dihasilkan benar-benar menangkap makna semantik; di sini korpus detik cuma ~58 kalimat pendek dan jsonplaceholder ~100 paragraf pendek. Jadi hasil `similar_by_key` di atas **tidak boleh ditafsirkan sebagai "makna sebenarnya"** - itu murni pola kemunculan bersama (co-occurrence) dalam korpus yang sangat terbatas ini, bukan representasi makna kata yang general.
- Khusus dataset jsonplaceholder, karena isinya memang teks *lorem ipsum* (bukan bahasa natural), hasil `similar_by_key('et')` secara linguistik tidak ada artinya sama sekali - ini jadi bukti nyata bahwa Word2Vec akan tetap menghasilkan angka berapa pun kualitas/kebermaknaan data sumbernya, sehingga **evaluasi kualitas data input sama pentingnya dengan memilih teknik feature engineering yang tepat**.
- Pada visualisasi UMAP dataset detik, kita bisa cek apakah kata-kata bertema serupa (misal `bijak`, `hidup`, `bahagia`) mengelompok berdekatan - kalau iya, itu indikasi model berhasil menangkap pola tematik meski dengan data terbatas.

## 5. FastText

### 5.1 Build Model FastText - dataset detik

In [27]:
#build model
model_ft_detik = gensim.models.FastText(sentences_detik, min_count=1,
                                         vector_size=100, window=5,
                                         min_n=3, max_n=6,
                                         sg=1, epochs=10)

In [28]:
# 2. save model
model_ft_detik.save('detik.fasttext')

# 3. load model
model_ft_detik = gensim.models.FastText.load('detik.fasttext')

# 4. model information
ft_detik = model_ft_detik.wv
ft_detik.index_to_key[:10]

['hidup',
 'orang',
 'bijak',
 'laku',
 'bahagia',
 'lihat',
 'syukur',
 'tuju',
 'cerdas',
 'senyum']

In [29]:
ft_detik.vectors.shape

(166, 100)

In [30]:
ft_detik.similar_by_key('bijak')

[('bicara', 0.2527835965156555),
 ('capai', 0.2223869115114212),
 ('puluh', 0.21867816150188446),
 ('cerdas', 0.1997014731168747),
 ('alah', 0.1720825433731079),
 ('bimbing', 0.16616909205913544),
 ('tuju', 0.1536070555448532),
 ('semangat', 0.14314435422420502),
 ('pandang', 0.143019437789917),
 ('esok', 0.1408107876777649)]

### 5.2 Build Model FastText - dataset jsonplaceholder

In [31]:
model_ft_api = gensim.models.FastText(sentences_api, min_count=1,
                                      vector_size=100, window=5,
                                      min_n=3, max_n=6,
                                      sg=1, epochs=10)
model_ft_api.save('jsonplaceholder.fasttext')
model_ft_api = gensim.models.FastText.load('jsonplaceholder.fasttext')

ft_api = model_ft_api.wv
ft_api.index_to_key[:10]

['et',
 'aut',
 'qui',
 'ut',
 'est',
 'quia',
 'voluptatem',
 'voluptas',
 'omnis',
 'molestiae']

In [32]:
ft_api.similar_by_key('et')

[('laudantium', 0.9996206760406494),
 ('similique', 0.9996170997619629),
 ('occaecati', 0.9996102452278137),
 ('deleniti', 0.9996093511581421),
 ('reiciendis', 0.9996007084846497),
 ('praesentium', 0.9995980858802795),
 ('optio', 0.9995970129966736),
 ('soluta', 0.9995965957641602),
 ('repudiandae', 0.9995956420898438),
 ('corporis', 0.9995936155319214)]

### 5.3 Visualisasi FastText

In [33]:
X = UMAP().fit_transform(ft_detik.vectors)

df2_detik_ft = pd.DataFrame(X, columns=['umap1', 'umap2'])
df2_detik_ft['word'] = ft_detik.index_to_key

fig = px.scatter(df2_detik_ft, x='umap1', y='umap2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(height=800, title_text='FastText Visualization - detik')
fig.show()

In [34]:
X = UMAP().fit_transform(ft_api.vectors)

df2_api_ft = pd.DataFrame(X, columns=['umap1', 'umap2'])
df2_api_ft['word'] = ft_api.index_to_key

fig = px.scatter(df2_api_ft, x='umap1', y='umap2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(height=800, title_text='FastText Visualization - jsonplaceholder')
fig.show()

### Analisis FastText

- Bedanya dengan Word2Vec: FastText memecah tiap kata jadi **subword n-gram** (`min_n=3, max_n=6`), jadi kata yang belum pernah muncul di training data (*out-of-vocabulary*) tetap bisa diberi vektor perkiraan dari potongan subword-nya. Ini cocok untuk Bahasa Indonesia yang morfologinya kaya imbuhan (awalan/akhiran), walaupun data kita sudah di-stem di Exercise 4 sehingga variasi imbuhan sudah banyak berkurang.
- Karena korpus di sini kecil, keunggulan FastText untuk data besar/kata jarang belum terlalu kelihatan efeknya - keunggulan ini baru akan sangat terasa pada korpus yang jauh lebih besar dan beragam.
- Sama seperti Word2Vec, hasil pada dataset jsonplaceholder tetap tidak bisa diinterpretasikan secara linguistik karena sumber datanya memang teks dummy.

## 6. Ringkasan & Kesimpulan

In [35]:
ringkasan = pd.DataFrame({
    'Teknik': ['One-Hot Encoding', 'TF-IDF', 'Word2Vec', 'FastText'],
    'Jenis Representasi': ['Sparse (biner ada/tidak)', 'Sparse (bobot term)', 'Dense (vektor makna)', 'Dense (vektor makna + subword)'],
    'Mempertimbangkan frekuensi kata?': ['Tidak', 'Ya', 'Tidak langsung (hasil training)', 'Tidak langsung (hasil training)'],
    'Bisa tangani kata baru (OOV)?': ['Tidak', 'Tidak', 'Tidak', 'Ya (lewat subword)'],
    'Butuh korpus besar supaya hasil optimal?': ['Tidak', 'Tidak', 'Ya', 'Ya'],
})
ringkasan

,Teknik,Jenis Representasi,Mempertimbangkan frekuensi kata?,Bisa tangani kata baru (OOV)?,Butuh korpus besar supaya hasil optimal?
0,One-Hot Encoding,Sparse (biner ada/tidak),Tidak,Tidak,Tidak
1,TF-IDF,Sparse (bobot term),Ya,Tidak,Tidak
2,Word2Vec,Dense (vektor makna),Tidak langsung (hasil training),Tidak,Ya
3,FastText,Dense (vektor makna + subword),Tidak langsung (hasil training),Ya (lewat subword),Ya


**Kesimpulan:**

1. **One-Hot Encoding** paling sederhana tapi paling boros memori (sparsity > 85% di kedua dataset) dan tidak membedakan kata penting vs kata biasa.
2. **TF-IDF** jadi perbaikan langsung dari One-Hot: pada dataset detik, kata-kata dengan skor tertinggi memang relevan secara tematik, sedangkan pada dataset jsonplaceholder hasilnya tetap hanya pola statistik tanpa makna linguistik - membuktikan bahwa TF-IDF bekerja secara matematis murni, bukan memahami bahasa.
3. **Word2Vec dan FastText** menghasilkan representasi dense yang secara teori bisa menangkap makna semantik, tapi pada project ini ukuran korpusnya **terlalu kecil** (58 dan 100 dokumen pendek) untuk hasil yang betul-betul reliabel - ini keterbatasan utama yang perlu disadari, bukan kesalahan implementasi.
4. **FastText** secara arsitektur lebih unggul dari Word2Vec untuk menangani kata jarang/baru lewat subword, dan berpotensi lebih cocok untuk Bahasa Indonesia yang morfologinya kaya, meski keunggulan itu belum terlalu terasa pada data sekecil ini.
5. Dataset jsonplaceholder (teks dummy/lorem ipsum) dipertahankan sebagai perbandingan untuk menunjukkan bahwa **kualitas hasil feature engineering sangat bergantung pada kualitas & kebermaknaan data sumber**, bukan cuma pada teknik yang dipakai.